# 01. Stuff 요약

`Stuff`("채워 넣다")는 **문서 전체를 하나의 프롬프트에 그대로 넣어** 한 번에 요약하는 가장 단순한 방식입니다.

- 장점: 구현이 가장 간단하고, LLM 호출이 1번뿐이라 빠르고 저렴합니다.
- 단점: 문서가 LLM 의 컨텍스트 창보다 길면 사용할 수 없습니다. → 이 경우 02(Map-Reduce), 03(Map-Refine) 방식을 사용합니다.

| 노트북 | 방식 |
|---|---|
| **01** | **Stuff** - 전체 문서를 한 번에 요약 |
| 02 | Map-Reduce - 분할 요약 후 일괄 병합 |
| 03 | Map-Refine - 분할 요약 후 순차적으로 다듬기 |
| 04 | Chain of Density - 반복하며 엔티티 밀도를 높이는 요약 |
| 05 | Clustering-Map-Refine - 클러스터 대표 청크만 골라 요약 |

## 환경 설정

- API 키(`OPENAI_API_KEY`, `LANGSMITH_API_KEY` 등)는 `.env` 파일 또는 시스템 환경변수에서 읽어옵니다.
- `logging.langsmith()` 를 호출하면 실행 과정이 LangSmith 에 기록되어 체인 내부 동작을 확인할 수 있습니다.

In [1]:
# .env 파일에 저장된 API KEY 를 환경변수로 불러옵니다.
from dotenv import load_dotenv

load_dotenv()

# LangSmith 추적 설정 (프로젝트 이름: Summary)
from langchain_teddynote import logging

logging.langsmith("Summary")

LangSmith 추적을 시작합니다.
[프로젝트명]
Summary


## 데이터 로드

뉴스 기사 텍스트 파일을 불러옵니다. 윈도우에서는 한글이 깨지지 않도록 `encoding="utf-8"` 을 반드시 지정합니다.

In [2]:
from langchain_community.document_loaders import TextLoader

# 노트북이 ch03_chains 폴더에 있으므로 상위 폴더(../)의 data 를 참조합니다.
loader = TextLoader("../data/news.txt", encoding="utf-8")
docs = loader.load()  # 파일 1개 → Document 1개

print(f"총 글자수: {len(docs[0].page_content)}")
print("\n========= 앞부분 미리보기 =========\n")
print(docs[0].page_content[:500])

C:\Users\user\AppData\Local\Temp\ipykernel_4340\3292615501.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


총 글자수: 1708

========= 앞부분 미리보기 =========

제목: 
AI2, 상업 활용까지 자유로운 '진짜' 오픈 소스 LLM '올모' 출시

내용:
앨런AI연구소(AI2)가 완전한 오픈 소스 대형언어모델(LLM) '올모(OLMo)’를 출시했다. 데이터 수집, 학습, 배포의 전 과정을 투명하게 공개한 데다 상업적 사용까지 허용한 진정한 의미의 오픈 소스 LLM이라는 평가다.
벤처비트는 1일(현지시간) 비영리 민간 AI 연구기관인 AI2가 ‘최초의 진정한 오픈 소스 LLM 및 프레임워크’라고 소개한 ‘올모’를 출시했다고 보도했다. 
이에 따르면 올모는 모델 코드와 모델 가중치뿐만 아니라 훈련 코드, 훈련 데이터, 관련 툴킷 및 평가 툴킷도 제공한다. 이를 통해 모델이 어떻게 구축되었는지 심층적으로 분석, LLM의 작동 방식과 응답을 생성하는 원리를 더 잘 이해할 수 있다. 
올모 프레임워크는 70억 매개변수의 ‘올모 7B’ 등 4가지 변형 모델과 10억 매개변수의 ‘올모 1B’ 모델을 제공한다. 모델들은 훈련 데이터를 생성하는 코드를 포함해 


### 프롬프트 가져오기 (LangSmith Hub)

> ⚠️ **레거시 변경점**
> 예전 코드의 `from langchain import hub` / `hub.pull(...)` 은 LangChain 1.x 에서 `langchain_classic` 으로 밀려났고,
> 최신 `langsmith` 는 **다른 사람(owner)의 공개 프롬프트**를 기본적으로 막습니다.
> 그래서 `langsmith.Client().pull_prompt(..., dangerously_pull_public_prompt=True)` 로 직접 가져옵니다.
> (`teddynote` 튜토리얼 프롬프트처럼 **신뢰할 수 있는 프롬프트에만** 사용하세요.)

In [3]:
from langsmith import Client

client = Client()


def pull_prompt(name: str):
    """LangSmith Hub 에서 공개 프롬프트를 가져옵니다. (hub.pull 대체)"""
    return client.pull_prompt(name, dangerously_pull_public_prompt=True)

In [4]:
# 한국어 + 이모지 bullet 로 요약하도록 지시하는 프롬프트 (입력 변수: context)
prompt = pull_prompt("teddynote/summary-stuff-documents-korean")
prompt.pretty_print()

Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. Translate the summary into KOREAN if it is written in ENGLISH.
5. DO NOT translate any technical terms.
6. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:"



## Stuff 체인 만들기 (LCEL)

> ⚠️ **레거시 변경점**
> 예전에는 `create_stuff_documents_chain(llm, prompt)` 를 사용했지만, 이 함수는 LangChain 1.x 에서
> `langchain_classic` 으로 이동한 **레거시**입니다.
> 하는 일은 "문서들을 문자열로 이어 붙여 `{context}` 에 넣고 → 프롬프트 → LLM" 뿐이므로,
> 요즘은 아래처럼 **LCEL(`|` 파이프)** 로 직접 구성합니다.

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI


def format_docs(docs):
    """Document 리스트를 하나의 문자열로 합칩니다. (create_stuff_documents_chain 이 내부에서 하던 일)"""
    return "\n\n".join(doc.page_content for doc in docs)


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Document 리스트 → {"context": 합친 문자열} → 프롬프트 → LLM → 문자열
stuff_chain = (
    {"context": RunnableLambda(format_docs)}
    | prompt
    | llm
    | StrOutputParser()
)

`stream_response` 는 토큰이 생성되는 대로 출력해 주고, 전체 답변 문자열을 반환합니다.

In [6]:
from langchain_teddynote.messages import stream_response

answer = stream_response(stuff_chain.stream(docs), return_output=True)

- 🚀 앨런AI연구소(AI2)가 완전한 오픈 소스 LLM '올모(OLMo)'를 출시했다.  
- 📊 데이터 수집, 학습, 배포 과정을 투명하게 공개하고 상업적 사용을 허용한다.  
- 🔍 올모는 모델 코드, 가중치, 훈련 코드, 데이터 및 평가 툴킷을 제공한다.  
- 🧩 70억 매개변수의 '올모 7B'와 10억 매개변수의 '올모 1B' 모델을 포함한 4가지 변형 모델이 있다.  
- 📈 훈련 데이터는 AI2의 '돌마(Dolma)' 데이터 세트를 기반으로 하며, 3조개의 토큰을 특징으로 한다.  
- 📝 아파치 2.0 라이선스에 따라 상업적 활용에 제한이 없다.  
- 💡 올모는 상업용 제품과 동등한 성능을 보여주며, 일부 자연어 처리 벤치마크에서 우수한 성과를 기록했다.  
- 🌍 비영어권 언어에 대한 낮은 품질과 약한 코드 생성 기능이 제약 사항으로 지적되었다.  
- 🔧 AI2는 올모를 계속해서 향상할 계획이다.  
- 🌐 올모의 모든 리소스는 깃허브 및 허깅페이스에서 무료로 제공된다.  

### (참고) 레거시 방식

`langchain_classic` 패키지가 설치되어 있으면 예전 코드도 아래처럼 import 경로만 바꿔서 실행할 수 있습니다.

```python
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

stuff_chain = create_stuff_documents_chain(llm, prompt)
answer = stuff_chain.invoke({"context": docs})
```